# Exciton Hamiltonian Block-Encoding Benchmark

This notebook builds `F`, `W`, `V` tensors, constructs the new `ExcitonHamiltonianBlockEncoding`, and reports resource metrics (gate counts, wall-time, logical qubits).

In [1]:
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd

from qualtran.resource_counting import QECGatesCost, get_cost_value, get_bloq_callee_counts

repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'experiments' else Path.cwd().resolve()
src_dir = repo_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from exciton.benchmark_tensors import generate_f_tensor, generate_v_tensor
from block_encoding.exciton_hamiltonian_encoding import build_exciton_hamiltonian_block_encoding


In [2]:
# Parameters (edit here)
m = 2
D = 1
L_c = 8
R_c = 1
R_loc = 1
metric = 'chebyshev'
entry_bitsize = 20
epsilon = 0.0

a = 1.0
b = 1.0
c = 1.0

shape = (L_c,) * D
print({'m': m, 'D': D, 'L_c': L_c, 'R_c': R_c, 'R_loc': R_loc, 'entry_bitsize': entry_bitsize, 'epsilon': epsilon})


{'m': 2, 'D': 1, 'L_c': 8, 'R_c': 1, 'R_loc': 1, 'entry_bitsize': 20, 'epsilon': 0.0}


In [3]:
# Build oracle-style tensors for the three components.
F = generate_f_tensor(
    shape=shape, a=a, metric=metric, r_cut=R_loc, oracle_convention='row_oracle'
)
W = generate_v_tensor(
    shape=shape, a=a, b=b, c=c, metric=metric, r_loc=R_loc, r_c=R_c, oracle_convention='exchange'
)
V = generate_v_tensor(
    shape=shape, a=a, b=b, c=c, metric=metric, r_loc=R_loc, r_c=R_c, oracle_convention='direct'
)

if epsilon > 0:
    F = np.where(np.abs(F) < epsilon, 0.0, F)
    W = np.where(np.abs(W) < epsilon, 0.0, W)
    V = np.where(np.abs(V) < epsilon, 0.0, V)

# Rescale two-particle tables for entry-oracle requirement M in [0, 1].
alpha_w = float(np.max(np.abs(W)))
alpha_v = float(np.max(np.abs(V)))
W_enc = (W / alpha_w) if alpha_w > 0 else W.copy()
V_enc = (V / alpha_v) if alpha_v > 0 else V.copy()

print('F shape:', F.shape)
print('W shape:', W.shape, 'alpha_w:', alpha_w, 'encoded max:', float(np.max(np.abs(W_enc))))
print('V shape:', V.shape, 'alpha_v:', alpha_v, 'encoded max:', float(np.max(np.abs(V_enc))))


F shape: (8, 3)
W shape: (8, 8, 3, 3) alpha_w: 2.0 encoded max: 1.0
V shape: (8, 8, 3, 3) alpha_v: 2.0 encoded max: 1.0


In [4]:
t0 = time.perf_counter()
exciton = build_exciton_hamiltonian_block_encoding(
    num_pairs=m,
    D=D,
    L=L_c,
    R_c=R_c,
    R_loc=R_loc,
    F=F,
    W=W_enc,
    V=V_enc,
    entry_bitsize=entry_bitsize,
)
build_time_s = time.perf_counter() - t0

print('Built:', type(exciton).__name__)
print('Build wall-time [s]:', round(build_time_s, 4))
print('Signature:', exciton.signature)


Built: ExcitonHamiltonianBlockEncoding
Build wall-time [s]: 0.0001
Signature: Signature((Register(name='q', dtype=QBit(), _shape=(), side=<Side.THRU: 3>), Register(name='h_sel', dtype=BQUInt(bitsize=2, iteration_length=3), _shape=(), side=<Side.THRU: 3>), Register(name='f_sel', dtype=BQUInt(bitsize=2, iteration_length=4), _shape=(), side=<Side.THRU: 3>), Register(name='f_m', dtype=BQUInt(bitsize=2, iteration_length=3), _shape=(1,), side=<Side.THRU: 3>), Register(name='w_sel', dtype=BQUInt(bitsize=3, iteration_length=6), _shape=(), side=<Side.THRU: 3>), Register(name='w_m', dtype=BQUInt(bitsize=2, iteration_length=3), _shape=(1,), side=<Side.THRU: 3>), Register(name='w_l', dtype=BQUInt(bitsize=2, iteration_length=3), _shape=(1,), side=<Side.THRU: 3>), Register(name='v_sel', dtype=BQUInt(bitsize=2, iteration_length=4), _shape=(), side=<Side.THRU: 3>), Register(name='v_m', dtype=BQUInt(bitsize=2, iteration_length=3), _shape=(1,), side=<Side.THRU: 3>), Register(name='v_l', dtype=BQUInt(bit

In [5]:
def logical_qubits_from_signature(sig):
    return int(sum(int(reg.total_bits()) for reg in sig.lefts()))

def _gatecounts_to_dict(gc):
    if hasattr(gc, 'asdict'):
        d = gc.asdict()
    else:
        d = dict(vars(gc))
    # Common aggregate used in your other analyses.
    d['toffoli_like'] = int(d.get('toffoli', 0) + d.get('and_bloq', 0) + d.get('cswap', 0))
    return d

def benchmark_bloq(name, bloq):
    out = {
        'name': name,
        'logical_qubits': logical_qubits_from_signature(bloq.signature),
    }

    t0 = time.perf_counter()
    try:
        gc = get_cost_value(bloq, QECGatesCost())
        out.update(_gatecounts_to_dict(gc))
        out['qec_status'] = 'ok'
    except Exception as exc:
        out['qec_status'] = f'failed: {type(exc).__name__}'

    out['qec_wall_time_s'] = time.perf_counter() - t0

    # Optional fallback: call-graph size (ignoring decomposition failures).
    try:
        cg = get_bloq_callee_counts(bloq, ignore_decomp_failure=True)
        out['num_callees'] = len(cg)
    except Exception:
        out['num_callees'] = np.nan

    return out

In [6]:
rows = []
rows.append(benchmark_bloq('F_component', exciton.f_bloq))
rows.append(benchmark_bloq('W_component', exciton.w_bloq))
rows.append(benchmark_bloq('V_component', exciton.v_bloq))
rows.append(benchmark_bloq('Exciton_total', exciton))

df = pd.DataFrame(rows)
cols = [
    'name', 'logical_qubits', 'qec_status', 'toffoli_like', 'toffoli', 'and_bloq',
    'clifford', 'rotation', 'measurement', 'qec_wall_time_s', 'num_callees'
]
cols = [c for c in cols if c in df.columns]
display(df[cols].fillna(''))


,name,logical_qubits,qec_status,toffoli_like,and_bloq,clifford,rotation,measurement,qec_wall_time_s,num_callees
0,F_component,17,ok,86,68,481,24,68,0.018979,17
1,W_component,20,ok,674,614,4421,32,614,0.022360,19
2,V_component,19,ok,588,564,4151,28,564,0.015393,10
3,Exciton_total,32,ok,1476,1374,9295,88,1374,0.034225,14


In [7]:
# Optional: quick summary dict for downstream scripts
summary = {
    'params': {
        'm': m, 'D': D, 'L_c': L_c, 'R_c': R_c, 'R_loc': R_loc,
        'entry_bitsize': entry_bitsize, 'epsilon': epsilon,
    },
    'rescale': {'alpha_w': alpha_w, 'alpha_v': alpha_v},
    'build_wall_time_s': build_time_s,
    'rows': rows,
}
summary


{'params': {'m': 2,
  'D': 1,
  'L_c': 8,
  'R_c': 1,
  'R_loc': 1,
  'entry_bitsize': 20,
  'epsilon': 0.0},
 'rescale': {'alpha_w': 2.0, 'alpha_v': 2.0},
 'build_wall_time_s': 6.339999526971951e-05,
 'rows': [{'name': 'F_component',
   'logical_qubits': 17,
   'cswap': 18,
   'and_bloq': 68,
   'clifford': 481,
   'rotation': 24,
   'measurement': 68,
   'toffoli_like': 86,
   'qec_status': 'ok',
   'qec_wall_time_s': 0.018979200001922436,
   'num_callees': 17},
  {'name': 'W_component',
   'logical_qubits': 20,
   'cswap': 60,
   'and_bloq': 614,
   'clifford': 4421,
   'rotation': 32,
   'measurement': 614,
   'toffoli_like': 674,
   'qec_status': 'ok',
   'qec_wall_time_s': 0.022360199996910524,
   'num_callees': 19},
  {'name': 'V_component',
   'logical_qubits': 19,
   'cswap': 24,
   'and_bloq': 564,
   'clifford': 4151,
   'rotation': 28,
   'measurement': 564,
   'toffoli_like': 588,
   'qec_status': 'ok',
   'qec_wall_time_s': 0.015393000001495238,
   'num_callees': 10},
  {

In [11]:
L_c = 3
D = 2

generate_v_tensor((L_c,) * D, r_loc=1, r_c=1).shape

(9, 9, 9, 9)